# 🔬 SPV Colab Sync: MPI PyMeep + Harminv 共振モード解析ノートブック（`spv` / `spv_gpu` 条件統一版）

本ノートブックは、元のローカルCPU版（`spv/`）および GPU版（`spv_gpu/`）と**シミュレーション条件（解像度・PML・セルサイズ・構造対称性・時間長）および `summary.csv` 形式を揃えた** MPI PyMeep パッケージ `spv_colab_sync` を実行するための Google Colab ランチャーです。
実装コードはすべて GitHub リポジトリ [`kyouoku/colab_GPU`](https://github.com/kyouoku/colab_GPU.git) に格納されており、ノートブックからは設定変更とパッケージ関数呼び出しのみを行います。

### 主な統一ポイント（対 `spv_gpu` / `spv`）
- **解像度**: `type=6` で `55 / 80`、`type=3〜5` で `60 / 90`（`quality="standard"` 時）
- **セルサイズ・PML厚**: `spv_gpu` と同一の空気余白 (`dist_to_pml`)・PML厚 (`pml_w`)・偶数グリッド整合
- **対称性修正**: z非対称な `type=3, 4`（マイクロディスク＋犠牲層・基板）では `pz=0` (`'o'`) を自動適用
- **高速化維持**: MPI 並列・多点同時励起・減衰停止 (`1e-5`)・STEP 2.5/3 DFT統合はそのまま利用
- **共通 `summary.csv`**: `spv_gpu` 互換列（`sym, pz, freq, Q, Q_max, Q_iqr, |A|, n_detections, rank_score, profile_png`）と `spv_colab` 互換列（`parity, res, elapsed_sec`）を両方出力

## 1. MPI版 PyMeep (`condacolab`) の環境構築
- 初回実行時は `condacolab.install()` によりランタイムが自動再起動されます。再起動後、もう一度上から実行してください。

In [ ]:
# ==============================================================================
# 1. condacolab と MPI版 PyMeep (mpi_mpich) のセットアップ
# ==============================================================================
import shutil
import subprocess
import sys

def _find_meep_python():
    for cand in [shutil.which("python"), "/usr/local/bin/python", "/usr/local/bin/python3", sys.executable]:
        if cand:
            res = subprocess.run(
                [cand, "-c", "import meep as mp; print(mp.__version__)"],
                capture_output=True,
                text=True,
                check=False,
            )
            if res.returncode == 0:
                return cand, res.stdout.strip().splitlines()[-1]
    return None, None

def setup_condacolab_and_meep():
    try:
        import condacolab
        condacolab.check()
        print("[OK] condacolab is ready.")
    except Exception:
        print("[Install] Installing condacolab==0.1.* (runtime will restart automatically)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.*"])
        import condacolab
        condacolab.install()
        print("[Info] 再起動後、このセル1をもう一度実行して PyMeep を導入してください。")
        return

    py_exe, mp_ver = _find_meep_python()
    if py_exe is None:
        installer = shutil.which("mamba") or shutil.which("conda")
        if not installer:
            raise RuntimeError("mamba / conda が見つかりません。セル1を再実行するかランタイムを再起動してください。")
        print(f"[Install] Installing pymeep=*=mpi_mpich_* h5py pandas matplotlib via {installer}...")
        subprocess.check_call([
            installer, "install", "-y", "-c", "conda-forge",
            "pymeep=*=mpi_mpich_*", "h5py", "pandas", "matplotlib"
        ])
        py_exe, mp_ver = _find_meep_python()

    if py_exe is None:
        raise RuntimeError("PyMeep の導入後に import meep が確認できませんでした。")
    print(f"[OK] PyMeep ready (version: {mp_ver}, python: {py_exe})")

setup_condacolab_and_meep()

## 2. Google Drive のマウント & GitHub (`kyouoku/colab_GPU`) の取得

In [ ]:
# ==============================================================================
# 2. Google Drive マウント & GitHub から spv_colab_sync パッケージを取得
# ==============================================================================
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/kyouoku/colab_GPU.git"
MOUNT_GOOGLE_DRIVE = True  # False にするとローカルの ./spv_colab_sync_results に保存

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DEFAULT_OUT_DIR = "/content/drive/MyDrive/spv_colab_sync_results"
        print(f"✅ Google Drive mounted. Default output_dir: {DEFAULT_OUT_DIR}")
    except Exception as e:
        DEFAULT_OUT_DIR = os.path.abspath("./spv_colab_sync_results")
        print(f"ℹ️ Drive mount skipped ({e}). Using {DEFAULT_OUT_DIR}")
else:
    DEFAULT_OUT_DIR = os.path.abspath("./spv_colab_sync_results")

if os.path.isdir("spv_colab_sync") and os.path.isfile("spv_colab_sync/config.py"):
    REPO_DIR = os.path.abspath(".")
elif os.path.isdir("../spv_colab_sync") and os.path.isfile("../spv_colab_sync/config.py"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/colab_GPU" if os.path.isdir("/content") else os.path.abspath("./colab_GPU")
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import spv_colab_sync
import spv_colab_sync.config
import spv_colab_sync.pipeline
import spv_colab_sync.benchmark
importlib.reload(spv_colab_sync.config)
importlib.reload(spv_colab_sync.pipeline)
importlib.reload(spv_colab_sync.benchmark)
importlib.reload(spv_colab_sync)

NP_PHYSICAL, NPROC_LOGICAL, HOST_MEM_GB = spv_colab_sync.pipeline.detect_host_resources()
os.environ["NP"] = str(NP_PHYSICAL)
os.environ["SPV_OUT_DIR"] = DEFAULT_OUT_DIR
print("✅ spv_colab_sync loaded from:", spv_colab_sync.__file__)

## 3. 計算条件パラメータ（`cfg`）の設定と導出値プレビュー
`spv_gpu` の `colab_run.ipynb` と同じ構造・キー名で設定できます。

In [ ]:
# ==============================================================================
# 3. 計算条件 (cfg) の編集と自動導出パラメータの確認
# ==============================================================================
from spv_colab_sync.config import get_default_config, derive_parameters

cfg = get_default_config()
cfg.update(
    structure_type = 6,            # 3:microdisk, 4:rough_microdisk, 5:waveguide_disk, 6:singular_bowtie_phc, 0:2D_PhC
    structure = dict(
        n            = 3.4,
        a            = 0.45,
        r_ratio      = 0.26,
        slab_ratio   = 0.50,
        N_ring       = 8,
        twist_deg    = 3.5,
        core_r_ratio = 0.55,
        gap_ratio    = 0.02,
        bowtie_angle = 60.0,
    ),
    f_range           = (0.55, 0.75),
    parities          = ["+", "-"],
    quality           = "standard", # "fast" | "standard" | "high"
    # accuracy        = dict(coarse=2, fine=4), # 指定時は quality より優先
    n_modes_select    = 2,
    Q_expected        = 1e4,
    run_mode          = "ALL",      # "ALL" | "HARMINV" | "MODE"
    merge_refine_mode = True,
    output_dir        = DEFAULT_OUT_DIR,
)

derived = derive_parameters(cfg, verbose=True)
print("=== 自動導出された計算パラメータ (spv_gpu / spv 同期条件) ===")
for k in [
    "structure_type", "n", "period", "d_min",
    "res_harminv", "res_mode", "fcen", "df", "cutoff",
    "dist_to_pml", "pml_w_harminv", "sp_harminv", "sp_mode",
    "parities", "n_modes", "run_mode", "tmax_coarse", "merge_refine", "out_dir"
]:
    print(f"  {k:16s}: {derived[k]}")

## 4. MPI 並列パイプラインの実行 & 結果表示

In [ ]:
# ==============================================================================
# 4. MPI 並列シミュレーションの実行と出力結果の可視化
# ==============================================================================
import os
from IPython.display import Image, display, Markdown
from spv_colab_sync.pipeline import run_mpi_pipeline

result = run_mpi_pipeline(cfg=cfg)

print(f"\n📂 出力ディレクトリ: {result['output_dir']}")
display(Markdown("### 📊 共振モード一覧 (`summary.csv`)"))
display(result["summary_df"])

if not result["best_df"].empty:
    display(Markdown("### 📋 `best condition` (高解像度補正後)"))
    display(result["best_df"])

for png_path in result["epchk_pngs"][:3]:
    display(Markdown(f"#### 🧱 誘電率断面: `{os.path.basename(png_path)}`"))
    display(Image(filename=png_path))

for png_path in result["harminv_pngs"]:
    display(Markdown(f"#### 📈 Harminv 検出分布: `{os.path.basename(png_path)}`"))
    display(Image(filename=png_path))

for png_path in result["mode_profile_pngs"][:8]:
    display(Markdown(f"#### 🌐 モード断面: `{os.path.basename(png_path)}`"))
    display(Image(filename=png_path))

## 5. CPU vs GPU 乖離比較 & ベンチマーク（任意実行）
GitHub 上の `spv_colab_sync.benchmark` から直接呼び出します。

In [ ]:
# ==============================================================================
# 5. CPU (spv_colab_sync) vs GPU (spv_gpu) 乖離比較 / 各種ベンチマーク
# ==============================================================================
import os
from IPython.display import Image, display
from spv_colab_sync.benchmark import (
    compare_cpu_vs_gpu,
    run_scaling_benchmark,
    run_all_structures_draft_test,
)

# --- A. CPU版と GPU版の summary.csv 突合 ---
GPU_SUMMARY_CSV = "/content/drive/MyDrive/spv_results/summary.csv"  # 必要に応じて変更
CPU_SUMMARY_CSV = result.get("summary_csv") or os.path.join(DEFAULT_OUT_DIR, "summary.csv")

if os.path.exists(CPU_SUMMARY_CSV) and os.path.exists(GPU_SUMMARY_CSV):
    comp_df = compare_cpu_vs_gpu(
        cpu_summary_csv=CPU_SUMMARY_CSV,
        gpu_summary_csv=GPU_SUMMARY_CSV,
        out_dir=DEFAULT_OUT_DIR,
    )
    display(comp_df)
    comp_png = os.path.join(DEFAULT_OUT_DIR, "cpu_vs_gpu_comparison.png")
    if os.path.exists(comp_png):
        display(Image(filename=comp_png))
else:
    print("ℹ️ 比較対象の GPU summary.csv が見つからないためスキップしました:", GPU_SUMMARY_CSV)

# --- B. MPI スケーリング / 全構造 Draft 検証 (必要なときだけ True に変更) ---
RUN_SCALING = False
RUN_ALL_DRAFT = False
if RUN_SCALING:
    display(run_scaling_benchmark())
if RUN_ALL_DRAFT:
    display(run_all_structures_draft_test())